# Differentially Private DNN - Gallstone Dataset\n\nTrains a Differentially Private DNN using Opacus (DP-SGD) on the Gallstone dataset.


## 1. Import Libraries


In [1]:
import time
import numpy as np
import pandas as pd
import pickle
import json
import os
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Shared metric collector (repository root): logs training accuracy,
# balanced accuracy and AUROC alongside the metrics already tracked below.
import os as _os
import sys as _sys
_REPO_ROOT = _os.path.abspath(_os.path.join('..', '..'))
if _REPO_ROOT not in _sys.path:
    _sys.path.insert(0, _REPO_ROOT)
from metrics_utils import ExtraMetrics


W0810 11:10:38.715000 37483 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Using device: cpu


## 2. Load Preprocessed Data


In [2]:
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

# Adaptive DP-SGD batch size (D13): scales with training-set size,
# clamped to [16, 256] so small datasets keep enough steps per epoch
# and large ones stay within Opacus' preferred large-batch regime.
n_train = X_train.shape[0]
batch_size = min(256, max(16, n_train // 8))
print(f"[D13] GALLSTONE: n_train={n_train} -> batch_size={batch_size}")
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


[D13] GALLSTONE: n_train=255 -> batch_size=31


## 3. Load Baseline Results


In [3]:
try:
    baseline_df = pd.read_csv('output/baseline_dnn_results.csv')
    baseline_accuracy = baseline_df['accuracy'].values[0]
    baseline_f1 = baseline_df['f1_score'].values[0]
except FileNotFoundError:
    baseline_accuracy = None
    baseline_f1 = None


## 4. Define DNN Architecture


In [4]:
class DPDNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_sizes=[64, 32], num_classes=2, dropout=0.3):
        super(DPDNNClassifier, self).__init__()
        layers = []
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

input_size = X_train.shape[1]
hidden_sizes = [64, 32]
num_classes = 2
dropout = 0.3
learning_rate = 0.001
num_epochs = 50


## 5. Training and Evaluation Functions


In [5]:
def train_dp_model(model, loader, criterion, optimizer, privacy_engine, epochs, dev, delta):
    model.train()
    for epoch in range(epochs):
        for inputs, labels in loader:
            inputs, labels = inputs.to(dev), labels.to(dev)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

def evaluate_model(model, loader, device):
    """Return (labels, predictions, class probabilities) for `loader`.

    The probabilities are what AUROC needs; predictions and labels are
    unchanged from the original implementation.
    """
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
            all_preds.extend(torch.max(outputs, 1)[1].cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


## 6. Sweep Epsilon


In [6]:
N_RUNS = 30
max_grad_norm = 1.0
delta = 1e-5
epsilon_values = [0.1, 0.2, 0.4, 0.8, 1.0, 2.0, 4.0, 8.0, 10.0]

results = {
    'epsilon': [], 'accuracy_mean': [], 'accuracy_std': [],
    'f1_score_mean': [], 'f1_score_std': [],
    'precision_mean': [], 'recall_mean': [],
}

all_results = {
    'model_type': 'Differentially Private DNN (DP-SGD via Opacus)',
    'dataset': 'Gallstone',
    'n_runs_per_epsilon': N_RUNS,
    'epsilon_results': {}
}

print(f"Training DP-DNN across {len(epsilon_values)} epsilon values...")
print('='*80)

for target_eps in epsilon_values:
    run_accs, run_f1s, run_precisions, run_recalls = [], [], [], []

    extra = ExtraMetrics()
    for run in range(N_RUNS):
        seed = run * 10 + 42
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)

        train_loader_temp = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        model = DPDNNClassifier(input_size, hidden_sizes, num_classes, dropout).to(device)
        model = ModuleValidator.fix(model)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        privacy_engine = PrivacyEngine()
        model, optimizer, train_loader_temp = privacy_engine.make_private_with_epsilon(
            module=model,
            optimizer=optimizer,
            data_loader=train_loader_temp,
            target_epsilon=target_eps,
            target_delta=delta,
            epochs=num_epochs,
            max_grad_norm=max_grad_norm,
        )

        if isinstance(optimizer.noise_multiplier, list):
            optimizer.noise_multiplier = optimizer.noise_multiplier[0]

        _t0 = time.perf_counter()
        train_dp_model(model, train_loader_temp, criterion, optimizer, privacy_engine, num_epochs, device, delta)
        train_time = time.perf_counter() - _t0
        y_true, y_pred, y_prob = evaluate_model(model, test_loader, device)

        run_accs.append(accuracy_score(y_true, y_pred))
        extra.add(y_true, y_pred, y_score=y_prob, train_time=train_time)
        run_f1s.append(f1_score(y_true, y_pred, zero_division=0))
        run_precisions.append(precision_score(y_true, y_pred, zero_division=0))
        run_recalls.append(recall_score(y_true, y_pred, zero_division=0))
        if run == 0:
            import sys
            import os
            sys.path.append(os.path.abspath(os.path.join('..', '..')))
            from model_exporter import export_model
            export_model(model, f'output/model/dp_dnn_model_eps_{target_eps}.pt')

    acc_mean, acc_std = np.mean(run_accs), np.std(run_accs)
    f1_mean, f1_std = np.mean(run_f1s), np.std(run_f1s)

    results['epsilon'].append(target_eps)
    # Merge in the added metrics; keys the notebook already tracks are skipped.
    for _key, _value in extra.summary().items():
        results.setdefault(_key, []).append(_value)
    results['accuracy_mean'].append(acc_mean)
    results['accuracy_std'].append(acc_std)
    results['f1_score_mean'].append(f1_mean)
    results['f1_score_std'].append(f1_std)
    results['precision_mean'].append(np.mean(run_precisions))
    results['recall_mean'].append(np.mean(run_recalls))

    all_results['epsilon_results'][str(target_eps)] = {
        'epsilon': float(target_eps), 'n_runs': N_RUNS,
        'accuracy_mean': acc_mean, 'accuracy_std': acc_std,
        'f1_mean': f1_mean, 'f1_std': f1_std,
        'precision_mean': np.mean(run_precisions),
        'recall_mean': np.mean(run_recalls),
        'per_run_accuracies': run_accs, 'per_run_f1s': run_f1s
    }
    all_results['epsilon_results'][str(target_eps)].update(extra.summary())
    all_results['epsilon_results'][str(target_eps)].update(extra.per_run())

    print(f"  ε={target_eps:5.1f} | Acc: {acc_mean:.4f}±{acc_std:.4f} | F1: {f1_mean:.4f}±{f1_std:.4f}")

print('='*80)


Training DP-DNN across 9 epsilon values...


PyTorch model successfully exported to output/model/dp_dnn_model_eps_0.1.pt


  ε=  0.1 | Acc: 0.5021±0.0257 | F1: 0.3954±0.2869


PyTorch model successfully exported to output/model/dp_dnn_model_eps_0.2.pt


  ε=  0.2 | Acc: 0.5047±0.0246 | F1: 0.3884±0.2883


...

  ε=  4.0 | Acc: 0.5813±0.0661 | F1: 0.3768±0.2863


PyTorch model successfully exported to output/model/dp_dnn_model_eps_8.0.pt


  ε=  8.0 | Acc: 0.6000±0.0605 | F1: 0.3982±0.2320


PyTorch model successfully exported to output/model/dp_dnn_model_eps_10.0.pt


  ε= 10.0 | Acc: 0.6047±0.0549 | F1: 0.4113±0.2153


## 7. Save Results


In [7]:
os.makedirs('output', exist_ok=True)
results_df = pd.DataFrame(results)

if baseline_accuracy is not None:
    results_df['ACL'] = 1 - (results_df['accuracy_mean'] / baseline_accuracy)
    results_df['accuracy_loss_pct'] = results_df['ACL'] * 100

results_df.to_csv('output/dp_dnn_results.csv', index=False)

# Attach the run count and raw per-run values that the sweep cell collected in
# `all_results` so the JSON report is a superset of the CSV.
_per_run_by_eps = {
    float(_k): _v for _k, _v in all_results['epsilon_results'].items()
}
_result_records = results_df.to_dict(orient='records')
for _record in _result_records:
    _entry = _per_run_by_eps.get(float(_record['epsilon']), {})
    _record.update({
        _k: _v for _k, _v in _entry.items()
        if _k == 'n_runs' or _k.startswith('per_run_')
    })

results_json = {
    "model_type": "DP-DNN",
    "dataset": "Gallstone",
    "privacy_mechanism": "DP-SGD (Opacus)",
    "parameters": {
        "architecture": hidden_sizes,
        "input_features": input_size,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "max_grad_norm": max_grad_norm,
        "delta": delta,
        "epochs": num_epochs,
        "batch_size": batch_size,
        "n_runs": N_RUNS
    },
    "results": _result_records,
    "baseline": {
        "accuracy": baseline_accuracy,
        "f1_score": baseline_f1
    } if baseline_accuracy is not None else None
}

with open('output/dp_dnn_report.json', 'w') as f:
    json.dump(results_json, f, indent=4)

print("Results saved.")


Results saved.
